# TempoVul: Artifact Generation and Evaluation Pipeline

Generates the four stage-specific artifacts for each sample (Requirements, Design, Implementation, Testing) and evaluates all five LLMs across all four SDLC stages, matching Sections 3.4 and 3.6 of the paper.

Stage 1 and Stage 2 artifacts are LLM-generated (Gemini 1.5 Pro) from a fixed template, then manually verified (Section 3.4). Stage 3 is the sample's source code. Stage 4 combines the source code with static analysis findings; see `03_stage4_correction_and_analysis.ipynb` for how the findings summary is built and incorporated.

**Inputs required:** `tempovul_base_dataset.csv` (from notebook 1), and access to the Gemini API (for artifact generation) and Hugging Face model weights (for LLM evaluation).


## 1. Generate Stage 1 and Stage 2 artifacts

In [ ]:
import pandas as pd
import time
from datetime import datetime

df = pd.read_csv('tempovul_base_dataset.csv')
print(f"Loaded {len(df)} samples")

def generate_requirements(code_sample):
    """Stage 1: natural language requirements specification."""
    prompt = f"""Given this C/C++ code, generate a concise requirements specification.

Code:
{code_sample[:1000]}

Format: "This function [purpose]. It should [behavior]. Security requirements: [constraints]."
Keep under 100 words."""
    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error: {e}"

def generate_design(code_sample, requirements):
    """Stage 2: data flow and trust boundary design specification."""
    prompt = f"""Given this code and requirements, generate a design specification.

Requirements: {requirements}

Code:
{code_sample[:1000]}

Include: Input (trust level), Output, Data flow, Trust boundaries.
Keep under 150 words."""
    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# Artifact generation loop. Stage 4's findings summary is added separately,
# see 03_stage4_correction_and_analysis.ipynb.

for col in ['stage1_requirements', 'stage2_design', 'stage3_implementation', 'stage4_testing']:
    if col not in df.columns:
        df[col] = ''

start_time = datetime.now()

for idx in range(len(df)):
    row = df.iloc[idx]

    if idx % 10 == 0:
        elapsed = (datetime.now() - start_time).total_seconds() / 60
        print(f"[{idx}/{len(df)}] {elapsed:.1f}m elapsed")

    if df.at[idx, 'stage1_requirements'] == '':
        reqs = generate_requirements(row['code'])
        df.at[idx, 'stage1_requirements'] = reqs
        time.sleep(2)

        design = generate_design(row['code'], reqs)
        df.at[idx, 'stage2_design'] = design
        time.sleep(2)

    df.at[idx, 'stage3_implementation'] = row['code']
    df.at[idx, 'stage4_testing'] = row['code'] + "\n\n[Static analysis placeholder]"

    if (idx + 1) % 50 == 0:
        df.to_csv('tempovul_with_artifacts_checkpoint.csv', index=False)

df.to_csv('tempovul_with_artifacts_final.csv', index=False)
print("Saved: tempovul_with_artifacts_final.csv")

## 2. Quality check

Manual verification of 50 random samples (12.5% of the dataset) achieved Cohen's kappa = 0.89 for inter-rater agreement on artifact quality, following a 30-sample pilot study (Section 3.4).

In [ ]:
df_final = pd.read_csv('tempovul_with_artifacts_final.csv')

print(f"Total samples: {len(df_final)}")
print(f"Vulnerable: {(df_final['label'] == 1).sum()}, Safe: {(df_final['label'] == 0).sum()}")
print()
for stage in ['stage1_requirements', 'stage2_design', 'stage3_implementation', 'stage4_testing']:
    complete = (df_final[stage] != '').sum()
    print(f"{stage}: {complete}/{len(df_final)} complete")

## 3. LLM evaluation

The stage prompt templates below are the final versions used to produce the paper's reported results (Section 3.6 / Appendix D). All five models are evaluated under identical zero-shot prompting.

See the paper's Section 3.7 for the instruction-following reliability finding: parseable output rates varied substantially across models, from 99.8% (Mistral-Instruct) to 16.9% (StarCoder2, a base rather than instruction-tuned model). Unparseable responses default to a negative prediction.

In [ ]:
EVALUATION_PROMPTS = {
    'stage1': """You are a security requirements analyst. Analyze these security requirements:

{artifact}

Determine if this requirement specification contains security vulnerabilities. Respond with ONLY valid JSON:
{{"vulnerable": true/false, "cwe": "CWE-XXX", "reasoning": "brief explanation"}}

JSON:""",

    'stage2': """You are a security architect. Review this system design:

{artifact}

Determine if this design has security vulnerabilities. Respond with ONLY valid JSON:
{{"vulnerable": true/false, "cwe": "CWE-XXX", "reasoning": "brief explanation"}}

JSON:""",

    'stage3': """You are a security code reviewer. Analyze this code:

{artifact}

Determine if this code contains security vulnerabilities. Respond with ONLY valid JSON:
{{"vulnerable": true/false, "cwe": "CWE-XXX", "reasoning": "brief explanation"}}

JSON:""",

    'stage4': """You are a security analyst reviewing test results and static analysis findings:

{artifact}

Determine if vulnerabilities are present. Respond with ONLY valid JSON:
{{"vulnerable": true/false, "cwe": "CWE-XXX", "reasoning": "brief explanation"}}

JSON:"""
}

print("Stage prompts defined for all 4 SDLC stages")

In [ ]:
def extract_json(text):
    """Finds the first complete, balanced JSON object in model output."""
    text = text.strip()
    start = text.find('{')
    if start == -1:
        return None
    brace_count = 0
    in_string = False
    escape_next = False
    for i in range(start, len(text)):
        char = text[i]
        if char == '"' and not escape_next:
            in_string = not in_string
        elif char == '\\' and in_string:
            escape_next = True
            continue
        if not in_string:
            if char == '{':
                brace_count += 1
            elif char == '}':
                brace_count -= 1
                if brace_count == 0:
                    return text[start:i+1]
        escape_next = False
    return None

def clean_for_csv(text):
    if not isinstance(text, str):
        return str(text)
    return ' '.join(text.split())

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_MAP = {
    "codellama": "codellama/CodeLlama-7b-Instruct-hf",
    "starcoder2": "bigcode/starcoder2-7b",
    "deepseek": "deepseek-ai/deepseek-coder-7b-instruct-v1.5",
    "mistral": "mistralai/Mistral-7B-Instruct-v0.3",
    "wizardcoder": "WizardLM/WizardCoder-15B-V1.0"
}

def evaluate_with_model(model, tokenizer, artifact, stage):
    try:
        prompt = EVALUATION_PROMPTS[stage].format(artifact=artifact[:2000])
        full_prompt = f"""{prompt}

Respond with ONLY valid JSON starting with {{

JSON:"""
        device = next(model.parameters()).device
        inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=2048)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=250, temperature=0.1,
                do_sample=False, pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response[len(full_prompt):].strip()

        json_str = extract_json(response)
        if json_str:
            return json.loads(json_str)
        return {"vulnerable": False, "cwe": "PARSE_ERROR", "reasoning": response[:200]}
    except Exception as error:
        return {"vulnerable": False, "cwe": "ERROR", "reasoning": str(error)[:200]}

# Run per model, e.g.:
# model_name = MODEL_MAP["mistral"]
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=BitsAndBytesConfig(load_in_4bit=True), device_map="auto")
# then loop over df, all 4 stages, calling evaluate_with_model(...)
# See src/utils/ for the full batch evaluation and PBS job scripts used on the
# Argonne Leadership Computing Facility Polaris cluster.

Next: see `03_stage4_correction_and_analysis.ipynb` for the Stage 4 artifact correction and the full corrected results analysis.